In [ ]:
# Task 1: Superstore Sales Data Cleaning and Exploratory Data Analysis
#**Internship:** ApexPlanet Data Analytics  
#**Dataset:** Superstore Sales  
#**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Plotly  
#**Objective:** Clean the dataset, explore patterns, create visualizations, and identify five business insights.#

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
raw_path = Path("/Users/apple/Downloads/superstore.csv")
df = pd.read_csv(raw_path)

df.head()

In [ ]:
print("Rows and columns:", df.shape)

display(df.head())
display(df.tail())

df.info()
display(df.describe(include="all").T)

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print(df.columns.tolist())

In [ ]:
print("Missing values before cleaning:")
display(df.isnull().sum().sort_values(ascending=False))

print("Duplicate rows before cleaning:", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates().copy()

for date_column in ["order_date", "ship_date"]:
    if date_column in df.columns:
        df[date_column] = pd.to_datetime(df[date_column], errors="coerce")

for numeric_column in ["sales", "profit", "discount", "quantity"]:
    if numeric_column in df.columns:
        df[numeric_column] = pd.to_numeric(df[numeric_column], errors="coerce")

for category_column in ["category", "sub_category", "segment", "region", "state", "city"]:
    if category_column in df.columns:
        df[category_column] = df[category_column].astype("category")

df.info()

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns
df[numeric_columns] = df[numeric_columns].fillna(df[numeric_columns].median())

df = df.dropna(subset=["order_date", "sales", "profit"])

print("Missing values after cleaning:")
display(df.isnull().sum().sort_values(ascending=False))

In [ ]:
def remove_iqr_outliers(dataframe, column):
    q1 = dataframe[column].quantile(0.25)
    q3 = dataframe[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    return dataframe[
        dataframe[column].between(lower_bound, upper_bound)
    ].copy()

before_rows = len(df)

for column in ["sales", "profit"]:
    if column in df.columns:
        df = remove_iqr_outliers(df, column)

after_rows = len(df)

print(f"Rows before outlier handling: {before_rows}")
print(f"Rows after outlier handling: {after_rows}")
print(f"Rows removed: {before_rows - after_rows}")

In [ ]:
df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.month
df["order_month_name"] = df["order_date"].dt.month_name()

clean_path = Path("/Users/apple/Downloads/superstore_cleaned.csv")
df.to_csv(clean_path, index=False)

print(f"Cleaned dataset saved to: {clean_path}")
print(f"Final dataset shape: {df.shape}")  

In [ ]:
# Exploratory Data Analysis

#This section uses summaries and visualizations to identify patterns in sales, profit, product categories, regions, and time.

In [ ]:
display(df[["sales", "profit", "discount", "quantity"]].describe().T)

print("Total sales:", round(df["sales"].sum(), 2))
print("Total profit:", round(df["profit"].sum(), 2))
print("Average order sales:", round(df["sales"].mean(), 2))

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df["sales"], bins=30, kde=True, color="steelblue")
plt.title("Distribution of Sales")
plt.xlabel("Sales")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x=df["profit"], color="orange")
plt.title("Distribution of Profit")
plt.xlabel("Profit")
plt.tight_layout()
plt.show()

In [ ]:
category_sales = (
    df.groupby("category", observed=True)["sales"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))
sns.barplot(x=category_sales.index, y=category_sales.values, hue=category_sales.index, legend=False)
plt.title("Total Sales by Category")
plt.xlabel("Category")
plt.ylabel("Total Sales")
plt.tight_layout()
plt.show()

In [ ]:
region_profit = (
    df.groupby("region", observed=True)["profit"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))
sns.barplot(x=region_profit.index, y=region_profit.values, hue=region_profit.index, legend=False)
plt.title("Total Profit by Region")
plt.xlabel("Region")
plt.ylabel("Total Profit")
plt.tight_layout()  
plt.show()  

In [ ]:
numeric_data = df[["sales", "profit", "discount", "quantity"]]

plt.figure(figsize=(8, 6))
sns.heatmap(numeric_data.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between Numeric Variables")
plt.tight_layout()
plt.show()

In [ ]:
monthly_sales = (
    df.groupby(df["order_date"].dt.to_period("M"))["sales"]
    .sum()
)

monthly_sales.index = monthly_sales.index.to_timestamp()

plt.figure(figsize=(12, 6))
plt.plot(monthly_sales.index, monthly_sales.values, marker="o", color="green")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

1. **Category performance:** [Best category] generated the highest sales of approximately [amount], while [lowest category] had the lowest sales.

2. **Regional profitability:** The [best region] region produced the highest profit, suggesting that its products, customers, or pricing strategy deserve closer study.

3. **Discount impact:** The correlation between discount and profit was [value]. This suggests that larger discounts [were / were not] associated with lower profitability.

4. **Sales concentration:** The histogram showed that most orders were in the [low / medium] sales range, while a smaller number of orders contributed very high sales values.

5. **Time trend:** Sales were strongest in [month/year or period] and weakest in [month/year or period], which may indicate seasonality and can inform inventory planning.